In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv
from google.cloud import bigquery

from src.utils.paths import get_project_root

In [2]:
load_dotenv()
project_root = get_project_root()


Project root: /home/didac/VSCode/video-games-data-pipeline


In [8]:
credentials_relative_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
print(f"Credentials path: {credentials_relative_path}")
if not credentials_relative_path:
    raise ValueError("GOOGLE_APPLICATION_CREDENTIALS not found in .env")

credentials_path = project_root / credentials_relative_path
if not credentials_path.exists():
    raise FileNotFoundError(f"GCP credentials file not found: {credentials_path}")
print(f"Using GCP credentials from: {credentials_path}")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path)

Credentials path: credentials/video-games-rawg-data-gcp-key.json
Using GCP credentials from: /home/didac/VSCode/video-games-data-pipeline/credentials/video-games-rawg-data-gcp-key.json


In [9]:
client = bigquery.Client()
print(f"Connected to GCP project: {client.project}")

Connected to GCP project: video-games-rawg-data


In [10]:
dataset_id = "rawg_games"
table_name = "games"

table_id = f"{client.project}.{dataset_id}.{table_name}"
print(f"Target table: {table_id}")

Target table: video-games-rawg-data.rawg_games.games


In [12]:
parquet_file = project_root / "data" / "processed" / "games.parquet"
print(f"Parquet file: {parquet_file}")
if not parquet_file.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_file}")

Parquet file: /home/didac/VSCode/video-games-data-pipeline/data/processed/games.parquet


In [16]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

    with parquet_file.open("rb") as source_file:
        job = client.load_table_from_file(
            source_file,
            table_id,
            job_config=job_config,
        )

job.result()

LoadJob<project=video-games-rawg-data, location=europe-southwest1, id=56b2d202-5cd5-4f09-ae1c-07960ef4171b>

In [17]:
table = client.get_table(table_id)
print(f"Loaded {table.num_rows} rows into {table_id}")

Loaded 400 rows into video-games-rawg-data.rawg_games.games
